# Federated Learning with Multiple Devices using Socket Communication

This is a minimal implementation of **Horizontal Federated Learning (FedAvg)** using multiple processes or physical machines. Communication between the server and clients is achieved via `socket`-based TCP connections.

## File Structure
```
federated_server.py      ← Server (aggregates model weights)  
federated_client.py      ← Client (performs local training)  
model.py                 ← Shared PyTorch model definition  
```

## 1. model.py — Shared PyTorch Model

In [ ]:
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(2, 10),
            nn.ReLU(),
            nn.Linear(10, 2)
        )

    def forward(self, x):
        return self.fc(x)

## 2. federated_client.py — Client Code

In [ ]:
import socket, pickle, copy
import torch
import torch.optim as optim
import torch.nn.functional as F
from model import SimpleNN

HOST = 'localhost'  # Replace with server IP if needed
PORT = 8000

def local_train(model, X, y, epochs=5, lr=0.01):
    model = copy.deepcopy(model)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = F.cross_entropy(output, y)
        loss.backward()
        optimizer.step()
    return model.state_dict()

def run_client():
    model = SimpleNN()
    X = torch.randn(100, 2)
    y = (X[:, 0] + X[:, 1] > 0).long()

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))
        print("Connected to server.")

        for round in range(10):
            data = s.recv(10**6)
            global_weights = pickle.loads(data)
            model.load_state_dict(global_weights)

            updated_weights = local_train(model, X, y)
            s.sendall(pickle.dumps(updated_weights))
            print(f"Round {round+1} completed.")

if __name__ == '__main__':
    run_client()

## 3. federated_server.py — Server Code

In [ ]:
import socket, pickle, copy
import threading
import torch
from model import SimpleNN

HOST = 'localhost'  # Use your machine's IP if on different devices
PORT = 8000
NUM_CLIENTS = 2

clients = []

def average_weights(weight_list):
    avg_weights = copy.deepcopy(weight_list[0])
    for key in avg_weights.keys():
        for i in range(1, len(weight_list)):
            avg_weights[key] += weight_list[i][key]
        avg_weights[key] = avg_weights[key] / len(weight_list)
    return avg_weights

def client_thread(conn, addr, client_id, shared_weights, lock):
    for rnd in range(10):
        with lock:
            conn.sendall(pickle.dumps(shared_weights[0]))

        data = conn.recv(10**6)
        client_weights = pickle.loads(data)

        with lock:
            shared_weights[1].append(client_weights)

def run_server():
    global_model = SimpleNN()
    shared_weights = [global_model.state_dict(), []]
    lock = threading.Lock()

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((HOST, PORT))
        s.listen()
        print(f"Server listening on {HOST}:{PORT}")

        for i in range(NUM_CLIENTS):
            conn, addr = s.accept()
            print(f"Client {i} connected from {addr}")
            clients.append(threading.Thread(target=client_thread, args=(conn, addr, i, shared_weights, lock)))

        for t in clients:
            t.start()

        for rnd in range(10):
            while True:
                with lock:
                    if len(shared_weights[1]) == NUM_CLIENTS:
                        break

            with lock:
                shared_weights[0] = average_weights(shared_weights[1])
                shared_weights[1] = []

            print(f"Round {rnd+1} aggregation completed.")

        for t in clients:
            t.join()

if __name__ == '__main__':
    run_server()

## How to Run

1. Start the server:
```bash
python federated_server.py
```

2. Then launch each client (in separate terminals or devices):
```bash
python federated_client.py
```

## Running on Multiple Devices

To run this system across multiple physical machines (e.g., in the same LAN or Wi-Fi network):

1. **On the server machine**, find its local IP address using:
```bash
ipconfig      # on Windows
ifconfig      # on Linux/macOS
```
Look for an IP like `192.168.x.x`.

2. **On each client**, update the `HOST` line:
```python
HOST = '192.168.x.x'  # Replace with the server's actual IP
```

3. Make sure:
- Both server and clients are on the same network.
- Port 8000 is not blocked by a firewall.
- Python and required libraries are installed on all machines.

## Further Notes

This is a minimal educational prototype. For production-grade use, consider:

- gRPC / HTTP-based communication  
- TLS encryption  
- Secure Aggregation  
- Differential Privacy  
- Real frameworks: [Flower](https://flower.dev/), [FedML](https://fedml.ai/), [PySyft](https://github.com/OpenMined/PySyft)